# TP1 — ¿Qué es un mercado en este problema?

**Mentorería DiploDatos 2026** · Proyecto Final: Detección de Ofertas Hoteleras ID90Travel

**Fecha de entrega:** 31/07

---

## Objetivo

Antes de intentar detectar si un precio es una oferta, hay que resolver una pregunta más básica: **¿contra qué se compara ese precio?**

La respuesta no es obvia. No alcanza con decir "contra otros precios del mismo destino" — porque los precios en Las Vegas un viernes de diciembre no tienen nada que ver con los de un martes de febrero, ni los de un hotel económico con los de uno de lujo.

El TP1 es una **exploración abierta del dataset** para entender qué dimensiones producen grupos de observaciones donde los precios se comportan de forma homogénea — es decir, donde comparar tiene sentido.

## Preguntas de investigación

### 1. Definición de mercado: ¿contra qué se compara?

**El problema de la referencia**: para saber si un precio es una oferta necesitás un precio de referencia del mismo mercado. Un hotel en Las Vegas no compite con uno en Orlando — aunque ambos sean destinos turísticos norteamericanos. Pero, ¿Las Vegas y Henderson (ciudad a 30 km) son el mismo mercado? ¿Y Las Vegas y Miami?

**La pregunta de fondo es: ¿qué define que dos destinos pertenecen al mismo mercado hotelero?**

**El problema de la fragmentación**: el dataset contiene ~26.000 nombres de ciudades distintos, con distribución de observaciones muy desigual. El trabajo en turismo hace que se consoliden datos de múltiples proveedores con sus propias estandarizaciones. Esto lleva a tener registros que refieren a la misma ciudad bajo nombres distintos.

**El rol fundamental de `destination_with_nearest.csv`**: este archivo mapea cada ciudad a un `nearest_destination_id` — el ID de un destino canónico cercano que tenga masa suficiente de observaciones. Sin esta consolidación, el análisis carece de la masa estadística necesaria para que cualquier comparación de precios sea confiable.

**Por qué es una hipótesis y no una verdad**: el criterio de agrupación usado es proximidad geográfica — razonable, pero discutible. ¿Miami y Miami Beach comparten realmente la misma dinámica de precios, o tienen perfiles distintos que al mezclarlos se distorsionan mutuamente?

**El mercado va más allá de la geografía**: la temporada del año, el día de la semana, la categoría del hotel o la duración de la estadía pueden ser tan determinantes como la geografía — o más.

### 2. Definición de contexto: ¿qué condiciones hacen que dos precios sean comparables?

El precio de un hotel nunca se evalúa en el vacío. Una habitación a $80/noche en Miami puede ser una ganga en diciembre (temporada alta) y un precio elevado en febrero (temporada baja).

**¿Qué conjunto de variables define que dos búsquedas son suficientemente similares como para comparar sus precios?**

Dimensiones a considerar:
- **Temporal**: mes del año, semana del mes, día de la semana, días especiales o eventos
- **Composición del viaje**: noches de estadía, número de habitaciones, adultos, niños
- **Categoría del alojamiento**: ¿los hoteles más caros de un destino siguen sus propias dinámicas?
- **Condiciones del mercado**: ¿el número de hoteles disponibles (`avg_hotel_count`) afecta los precios observados?

*La definición de CONTEXTO determina directamente la calidad de cualquier algoritmo de detección.*

## Dimensiones candidatas a explorar

No es necesario explorar todas — lo importante es llegar a una **hipótesis razonada** sobre qué combinación produce segmentos internamente coherentes y suficientemente densos como para sostener un análisis estadístico.

| Dimensión | Variable(s) | Pregunta guía |
|-----------|-------------|---------------|
| Localización geográfica | `city`, `state`, `country`, `destination_final` | ¿Los precios son comparables entre destinos cercanos? |
| Día de la semana | derivada de `date_start` | ¿Los precios suben los fines de semana? ¿En todos los destinos por igual? |
| Temporada del año | `month` derivado de `date_start` | ¿La estacionalidad afecta igual a todos los mercados? |
| Categoría del hotel | segmentos de precio (`price_bucket`) | ¿Los hoteles económicos siguen el mismo ciclo que los de lujo? |
| Duración de la estadía | `nights` | ¿Las estadías largas tienen dinámica distinta? |
| Demanda | `count_repeated` | ¿Los destinos con alta demanda tienen precios más estables? |
| Oferta | `avg_hotel_count` | ¿Cuando colapsa la oferta, suben los precios de forma distinta? |

## Preguntas disparadoras

Algunas preguntas para guiar la exploración:

1. ¿Los precios suben los fines de semana?
2. ¿En todos los destinos por igual, o solo en los de ocio?
3. ¿Los hoteles económicos siguen el mismo ciclo estacional que los de lujo?
4. ¿Hay destinos donde la oferta disponible (`avg_hotel_count`) colapsa en ciertas fechas y eso afecta los precios de forma distinta al resto?
5. ¿Qué combinación de variables produce grupos con precios más homogéneos (menor varianza interna)?

## Datos disponibles

| Archivo | Descripción |
|---------|-------------|
| `datos_historicos_2024.csv` | ~2.5M registros de búsquedas hoteleras 2024 (~200 MB) |
| `datos_historicos_2025.csv` | ~2.5M registros de búsquedas hoteleras 2025 (~200 MB) |
| `destination_with_nearest.csv` | Mapping de ~26.000 ciudades a destinos canónicos |

**Importante**: cada fila no representa el precio de un hotel individual, sino el rango estadístico de *todos* los hoteles mostrados en esa búsqueda.

### Columnas principales del dataset

| Columna | Descripción |
|---------|-------------|
| `city`, `state`, `country`, `country_code` | Destino geográfico |
| `date_start`, `date_end`, `nights` | Fechas y duración de la estadía |
| `number_of_rooms`, `number_of_adults`, `number_of_kids` | Composición del viaje |
| `count_repeated` | Cantidad de veces que se realizó esta búsqueda (proxy de demanda) |
| `avg_hotel_count` | Promedio de hoteles disponibles mostrados (proxy de oferta) |
| `avg_price_average` | **Variable central**: precio promedio del conjunto de hoteles en USD |
| `max_price_high`, `min_price_low` | Rango de precios del mercado |

### Datos descargados en: `../data/`

## Entrega esperada

Notebook con:

1. **Visualizaciones** que muestren el comportamiento de los precios a lo largo de las distintas dimensiones
2. **Hipótesis documentada** sobre qué segmentación de mercado propone el grupo — y por qué
3. **Análisis de densidad**: ¿las segmentaciones propuestas generan suficientes observaciones por grupo para ser estadísticamente confiables?

---

---
# Exploración inicial · Setup y carga de datos

Las celdas a continuación replican los pasos clave del `exploratory_analysis.ipynb` para que puedas arrancar la exploración del TP1 directamente.

In [ ]:
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from auxiliary_functions import (
    load_all_historicals,
    load_destination_mapping,
    standardize_prices,
    expand_dates_dataframe,
    add_temporal_features,
    apply_destination_mapping,
)
import config

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)
print('Librerías importadas correctamente')
print(f'Directorio de datos: {config.DATA_DIR}')

### Carga de datos

Se carga una **muestra de 100.000 registros** para exploración rápida. Para el análisis completo, usar `load_all_historicals()`.

In [ ]:
archivos = sorted(glob.glob(str(config.DATA_DIR / config.HISTORICALS_FILE_PATTERN)))
if not archivos:
    raise FileNotFoundError(f'No se encontraron archivos en {config.DATA_DIR}')

df_raw = pd.read_csv(archivos[0], nrows=100_000)
print(f'Archivo:             {Path(archivos[0]).name}')
print(f'Registros cargados:  {len(df_raw):>12,}  (muestra de 100k)')
print(f'Columnas:            {len(df_raw.columns)}')
print()
print('Lista de columnas:')
for col in df_raw.columns:
    print(f'  - {col}  ({df_raw[col].dtype})')

In [ ]:
df_raw.head(3)

### Cobertura del dataset

In [ ]:
print('COBERTURA DEL DATASET')
print('=' * 45)
print(f"  Fecha inicio:       {df_raw['date_start'].min()}")
print(f"  Fecha fin:          {df_raw['date_start'].max()}")
print(f"  Países:             {df_raw['country'].nunique()}")
print(f"  Estados/Provincias: {df_raw['state'].nunique():,}")
print(f"  Ciudades:           {df_raw['city'].nunique():,}")
print()
print('DEMANDA (count_repeated = búsquedas agregadas)')
print('=' * 45)
print(f"  Total búsquedas:    {df_raw['count_repeated'].sum():,.0f}")
print(f"  Media por registro: {df_raw['count_repeated'].mean():.1f}")
print()
print('PRECIOS (avg_price_average)')
print('=' * 45)
df_raw['avg_price_average'].describe()

---
## ¿Qué destinos dominan el dataset?

In [ ]:
top_dest = (
    df_raw.groupby('city')['count_repeated']
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(13, 5))
top_dest.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 ciudades por volumen de búsquedas', fontsize=13)
ax.set_xlabel('Ciudad')
ax.set_ylabel('Búsquedas totales (count_repeated)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

top5_pct = top_dest.head(5).sum() / df_raw['count_repeated'].sum()
print(f'Top 5 ciudades concentran el {top5_pct:.1%} del total de búsquedas.')

---
## Normalización de precios

```
precio_normalizado = precio_total / (noches × habitaciones × (adultos + niños))
```

Esto produce un **precio por persona por habitación por noche**, comparable entre búsquedas de cualquier composición.

In [ ]:
df = df_raw.drop_duplicates()
df = standardize_prices(df)

print('Comparación precio total vs. precio normalizado:')
df[['avg_price_average', 'avg_price_average_std']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

clip_raw = df['avg_price_average'].quantile(0.99)
df['avg_price_average'].clip(upper=clip_raw).hist(
    bins=80, ax=axes[0], color='coral', edgecolor='white'
)
axes[0].set_title('Precio total (sin normalizar)')
axes[0].set_xlabel('USD')

clip_std = df['avg_price_average_std'].quantile(0.99)
df['avg_price_average_std'].clip(upper=clip_std).hist(
    bins=80, ax=axes[1], color='steelblue', edgecolor='white'
)
axes[1].set_title('Precio normalizado (por persona-hab-noche)')
axes[1].set_xlabel('USD')

plt.tight_layout()
plt.show()

---
## Expansión temporal y features

Cada registro representa una estadía multi-noche. Para análisis con resolución semanal, expandimos a observaciones diarias y agregamos `month` y `week_in_month`.

In [ ]:
registros_antes = len(df)
df = expand_dates_dataframe(df)
print(f'Registros antes:   {registros_antes:>12,}')
print(f'Registros después: {len(df):>12,}')
print(f'Factor expansión:  {len(df)/registros_antes:.2f}x')

df = add_temporal_features(df)
print(f'\nColumnas agregadas: month, week_in_month')
df[['date_start', 'date_end', 'nights', 'date', 'month', 'week_in_month']].head()

### Estacionalidad: precio promedio por mes

In [ ]:
precios_mes = (
    df.groupby('month')['avg_price_average_std']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(precios_mes['month'], precios_mes['avg_price_average_std'],
       color='steelblue', edgecolor='white')
ax.set_xlabel('Mes')
ax.set_ylabel('Precio normalizado promedio (USD)')
ax.set_title('Estacionalidad: precio promedio normalizado por mes')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'])
plt.tight_layout()
plt.show()

---
## Mapping de destinos

El mapping consolida ~26.000 nombres de ciudades en identificadores únicos de destino canónico.

In [ ]:
mapping_df = load_destination_mapping()
print(f'Referencias en el mapping: {len(mapping_df):,}')
print(f'Destinos canónicos únicos: {mapping_df["nearest_destination_id"].nunique():,}')
print()
mapping_df.head()

In [ ]:
df = apply_destination_mapping(df, mapping_df)

print('Resultado del mapping:')
print(f'  Destinos únicos finales:         {df["destination_final"].nunique():,}')
sin_mapeo = df['destination_final'].isna().sum()
print(f'  Sin mapeo (fallback a ciudad):   {sin_mapeo:,} registros ({sin_mapeo/len(df):.1%})')
con_mapeo = (~df['destination_final'].isna()).sum()
print(f'  Con mapeo canónico:              {con_mapeo:,} registros ({con_mapeo/len(df):.1%})')

---
## Arrancá tu exploración acá

Usá las variables `df` (con las features ya construidas) para explorar las dimensiones del TP1:
- `avg_price_average_std`: precio normalizado
- `destination_final`: destino canónico
- `month`, `week_in_month`: features temporales
- `count_repeated`: proxy de demanda
- `avg_hotel_count`: proxy de oferta
- `nights`: duración de estadía
- `number_of_adults`, `number_of_rooms`: composición del viaje

In [ ]:
# Arrancá acá con tu exploración
